In [4]:
import jax
import jax.numpy as jnp

import flax.linen as nn
from flax.training import train_state
import math
import optax
from dataclasses import dataclass
import numpy as np

In [5]:
import os
os.environ["JAX_PLATFORMS"] = "mps"

Data Generation Functions

In [6]:
VOCAB="0123456789+= "
MAX_SEQ_LEN=len("999+999=1998")+1

stoi={c:i for i,c, in enumerate(VOCAB)}
itos={i:c for c,i in stoi.items()}

PAD = stoi[" "]

def encode(text):
  ids=[]
  for i in text:
    ids.append(stoi[i])
  if len(ids)<MAX_SEQ_LEN:
    ids+=[PAD]*(MAX_SEQ_LEN-len(ids))
  return jnp.array(ids, dtype=jnp.int32)

def decode(arr):
  ints=[]
  for i in arr:
    #need to be changed to ints because jax will store individual values as arrays
    ints.append(itos[int(i)])
  return "".join(ints)

def create_sample(a,b):
  return encode(f"{a}+{b}={a+b}")

def create_batches(batch_size):
  a = np.random.randint(0,1000,size=(batch_size,))
  b = np.random.randint(0,1000,size=(batch_size,))
  data = jnp.array([create_sample(i,j) for i,j, in zip(a,b)])

  x = data[:,:-1]
  y = data[:,1:]

  return x,y

Transformer Implementation Functions



In [19]:
@dataclass(frozen=True)
class Config:
  d_model: int = 384
  n_heads: int = 6
  n_layers: int = 6
  vocab_size: int = len(VOCAB)
  max_seq_len: int = MAX_SEQ_LEN
  n_experts: int = 8
  use_moe: bool = False
  ff_d_ratio: int=4

class Attention(nn.Module):
  config: Config
  # use compact so we dont need to specify shapes aside from output shapes
  @nn.compact
  def __call__(self,x):
    B,T,D = x.shape
    qkv = nn.Dense(3*D, use_bias=False)(x)
    q,k,v = jnp.split(qkv, 3, axis=-1)

    head_dim = self.config.d_model//self.config.n_heads

    #reshape from BTD -> BTNH
    q = q.reshape(B,T,self.config.n_heads,head_dim)
    k = k.reshape(B,T,self.config.n_heads,head_dim)
    v = v.reshape(B,T,self.config.n_heads,head_dim)

    #swapping T <-> N so we can compute our matmuls over the sequences
    q = jnp.swapaxes(q, 1, 2)
    k = jnp.swapaxes(k, 1, 2)
    v = jnp.swapaxes(v, 1, 2)

    #Q (BNTH) dot K transposed (BNHT). Since Jax ignores the first 2 dims during matmuls, we end up with BNTT
    att = jnp.matmul(q,jnp.swapaxes(k,-1,-2))

    att = att/math.sqrt(head_dim)

    #triangle mask
    mask = jnp.tril(jnp.ones((T,T)))

    #values that are not 1 are set to neg inf, so when we softmask they essentially become 0
    att = jnp.where(mask[None,None,:,:]==1, att, -1e9)

    #softmax over keys
    att = jax.nn.softmax(att, axis=-1)

    #QK (BNTT) dot V (BNTH) -> BNTH
    out = jnp.matmul(att,v)

    #need to swap these axes (N and T)
    out = jnp.swapaxes(out,1,2)

    #Now we have BTNH, but we want BTD, so we need to combine N and H, since NH=D
    out = out.reshape(B,T,D)

    #Now we multiply by W out
    out = nn.Dense(D)(out)

    return out

class MLP(nn.Module):
  config: Config
  @nn.compact
  def __call__(self,x):
    #W in
    x = nn.Dense(4*self.config.d_model)(x)
    x = nn.gelu(x)
    #W out
    x = nn.Dense(self.config.d_model)(x)
    return x

class MOE(nn.Module):
  config: Config
  @nn.compact
  def __call__(self, x):
    E = self.config.n_experts
    ff_d = self.config.ff_d_ratio

    B,T,D = x.shape
    x_flat = x.reshape(B*T,D)
    router_logits = nn.Dense(E, use_bias=False)(x_flat) #B*T, E
    router_probs = jax.nn.softmax(router_logits, axis=-1) # softmax along axis E (experts)



    top_1_probs, top_1_indices = jax.lax.top_k(router_probs, k=1) #B*T, 1

    probs_flat = top_1_probs.reshape(-1) #B,T
    indices_flat = top_1_indices.reshape(-1) #B,T

    sort_idx = jnp.argsort(indices_flat) #no need to specify axis since indices are flattened

    #sorting everything by expert
    x_sorted = x_flat[sort_idx]
    probs_sorted = probs_flat[sort_idx]

    #In/Out weights
    W_in = self.param("W_in", nn.initializers.lecun_normal(), (E,D,ff_d*D)) #project to ff_d
    W_out = self.param("W_out", nn.initializers.lecun_normal(), (E,ff_d*D,D)) #collapse back to D

    group_size = jnp.bincount(indices_flat, length=E)

    h = jax.lax.ragged_dot(x_sorted, W_in, group_size)
    h = nn.gelu(h)
    out = jax.lax.ragged_dot(h, W_out, group_size)

    #multiply out by the sorted probs (gate)
    out = out * probs_sorted[:, None]

    #get unsorting idx list
    unsort_idx = jnp.argsort(sort_idx) #pretty neat because the sort_idx is a list containing all original indices

    out = out[unsort_idx]

    out = out.reshape(B,T,D) #go from (B*T, D) -> (B,T,D)

    return out

class Transformer(nn.Module):
  config: Config
  @nn.compact
  def __call__(self, x):
    #residual connections and layernorms before each attention and mlp layer
    x = x + Attention(self.config)(nn.LayerNorm()(x))
    if self.config.use_moe:
      x = x + MOE(self.config)(nn.LayerNorm()(x))
    else:
      x = x + MLP(self.config)(nn.LayerNorm()(x))
    return x

class LanguageModel(nn.Module):
  config:Config
  @nn.compact
  def __call__(self, tokens):
    _,T = tokens.shape
    #VD embedding for tokens
    tok_emb = nn.Embed(self.config.vocab_size, self.config.d_model)(tokens)
    #get pos emb params and set std to 0.02, as per gpt2 paper
    pos_emb = self.param("pos_emb", nn.initializers.normal(0.02), (self.config.max_seq_len, self.config.d_model))

    #cut off pos_emb because it may be longer than the sequence
    x = tok_emb+pos_emb[:T]

    #now we can send x through transformer blocks
    for _ in range(self.config.n_layers):
      x=Transformer(self.config)(x)

    #Final layer norm
    x = nn.LayerNorm()(x)

    #shape of x right now is BTD, we need BTV
    logits = nn.Dense(self.config.vocab_size)(x)

    return logits


Training

In [ ]:
config = Config

model = LanguageModel(config)

rng = jax.random.PRNGKey(42)

mock_data = jax.ShapeDtypeStruct((1,MAX_SEQ_LEN-1), dtype=jnp.int32)

#lazy init is supposed to be faster than using model.apply

params = model.lazy_init(rng, mock_data)

tx = optax.adamw(learning_rate=3e-4, weight_decay = 1e-2)

state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

def loss_fn(params, x, y):
  logits = model.apply(params,x)
  loss = optax.softmax_cross_entropy_with_integer_labels(logits,y)
  return loss.mean()


def train_step(state,x,y):
     loss, grads = jax.value_and_grad(loss_fn)(state.params,x,y)
     state = state.apply_gradients(grads=grads)
     return loss, state

#training loop
BATCH_SIZE=256
STEPS=5000

for i in range(STEPS):
  x,y = create_batches(BATCH_SIZE)
  loss, state = train_step(state,x,y)
  if i%100==0:
    print(f"{i}:{loss}")



In [22]:
def val_step(params, x, y):
    return loss_fn(params, x, y)

VAL_STEPS=5

def val_loop():
    losses=[]
    for i in range(VAL_STEPS):
        x,y = create_batches(BATCH_SIZE)
        losses.append(val_step(state.params, x, y))
    return sum(losses)/VAL_STEPS

In [37]:
import json


TOKENS_PER_STEP = BATCH_SIZE*MAX_SEQ_LEN

parameter_sweep = [1000000, 5000000, 10000000]

grid={}

token_multipliers = [0.01, 0.05, 0.1, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

for num_params in parameter_sweep:

    num_tokens = [num_params*token_multiplier for token_multiplier in token_multipliers]

    flops = [6*num_params*t for t in num_tokens]

    steps = [t // TOKENS_PER_STEP for t in num_tokens]


    runs_for_model = []
    for t,f,s in zip(num_tokens, flops, steps):
        runs_for_model.append({
            "tokens": t,
            "flops": f,
            "steps": max(1,s),
        })

    grid[f'params_{num_params}'] = runs_for_model

with open('dense_runs.json', 'w') as f:
    json.dump(grid, f)

In [38]:
for i in grid.keys():
    if i == 'params_1000000':
        config = Config(d_model=128, n_layers=5, n_heads=4)
    if i == 'params_5000000':
        config = Config(d_model=320, n_layers=4, n_heads=4)
    if i == 'params_10000000':
        config = Config(d_model=384, n_layers=6, n_heads=6)

    for run in grid[i]:

        model = LanguageModel(config)
        params = model.lazy_init(rng, mock_data)
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        steps = run['steps']

        print(f'steps: {steps}')

        for step_idx in range(int(steps)):
            x,y = create_batches(BATCH_SIZE)
            loss, state = train_step(state, x, y)
            if step_idx % 100 == 0:
                print(f'step {step_idx}: {loss}')

        print('done training')
        run['train_loss'] = loss.item()

        val_loss = val_loop()
        run['val_loss'] = val_loss.item()

        print('done validating')

        run['params']=sum(x.size for x in jax.tree_util.tree_leaves(params))  


        with open('dense_runs.json', 'w') as f:
                json.dump(grid, f)
        
        print('run saved')

steps: 3.0
step 0: 2.9953527450561523
done training
done validating
run saved
steps: 15.0
step 0: 3.061142683029175
done training
done validating
run saved
steps: 30.0
step 0: 3.039788007736206
done training
done validating
run saved
steps: 150.0
step 0: 3.0173046588897705
step 100: 1.5465950965881348
done training
done validating
run saved
steps: 300.0
step 0: 3.0226032733917236
step 100: 1.5464273691177368
step 200: 1.4528435468673706
done training
done validating
run saved
steps: 450.0
step 0: 3.0508546829223633
step 100: 1.5461336374282837
step 200: 1.4507783651351929
step 300: 1.4163411855697632
step 400: 1.4050754308700562
done training
done validating
run saved
steps: 600.0
step 0: 3.045987129211426
step 100: 1.5539222955703735
step 200: 1.455877661705017
step 300: 1.4180426597595215
step 400: 1.4065953493118286
step 500: 1.39557945728302
done training
done validating
run saved
steps: 751.0
step 0: 3.0231053829193115
step 100: 1.5581697225570679
step 200: 1.4588991403579712
step

In [39]:
import json


TOKENS_PER_STEP = BATCH_SIZE*MAX_SEQ_LEN

parameter_sweep = [1000000, 5000000, 10000000]

grid={}

token_multipliers = [0.01, 0.05, 0.1, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

for num_params in parameter_sweep:

    num_tokens = [num_params*token_multiplier for token_multiplier in token_multipliers]

    flops = [6*num_params*t for t in num_tokens]

    steps = [t // TOKENS_PER_STEP for t in num_tokens]


    runs_for_model = []
    for t,f,s in zip(num_tokens, flops, steps):
        runs_for_model.append({
            "tokens": t,
            "flops": f,
            "steps": max(1,s),
        })

    grid[f'params_{num_params}'] = runs_for_model

with open('moe_runs.json', 'w') as f:
    json.dump(grid, f)

In [40]:
for i in grid.keys():
    if i == 'params_1000000':
        config = Config(d_model=128, n_layers=5, n_heads=4, use_moe=True, n_experts=6)
    if i == 'params_5000000':
        config = Config(d_model=320, n_layers=4, n_heads=4, use_moe=True, n_experts=6)
    if i == 'params_10000000':
        config = Config(d_model=384, n_layers=6, n_heads=6, use_moe=True, n_experts=6)

    for run in grid[i]:

        model = LanguageModel(config)
        params = model.lazy_init(rng, mock_data)
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        steps = run['steps']

        print(f'steps: {steps}')

        for step_idx in range(int(steps)):
            x,y = create_batches(BATCH_SIZE)
            loss, state = train_step(state, x, y)
            if step_idx % 100 == 0:
                print(f'step {step_idx}: {loss}')

        print('done training')
        run['train_loss'] = loss.item()

        val_loss = val_loop()
        run['val_loss'] = val_loss.item()

        print('done validating')

        run['params']=sum(x.size for x in jax.tree_util.tree_leaves(params))  


        with open('moe_runs.json', 'w') as f:
                json.dump(grid, f)
        
        print('run saved')

steps: 3.0
step 0: 3.062520980834961
done training
done validating
run saved
steps: 15.0
step 0: 3.0544216632843018
done training
done validating
run saved
steps: 30.0
step 0: 3.0289087295532227
done training
done validating
run saved
steps: 150.0
step 0: 3.0444040298461914
step 100: 1.5506685972213745
done training
done validating
run saved
steps: 300.0
step 0: 3.0017378330230713
step 100: 1.5472081899642944
step 200: 1.4448342323303223
done training
done validating
run saved
steps: 450.0
step 0: 3.0177528858184814
step 100: 1.5520073175430298
step 200: 1.4539860486984253
step 300: 1.4149278402328491
step 400: 1.405504584312439
done training
done validating
run saved
steps: 600.0
step 0: 3.0452888011932373
step 100: 1.5451513528823853
step 200: 1.4503278732299805
step 300: 1.4217424392700195
step 400: 1.4084457159042358
step 500: 1.3909435272216797
done training
done validating
run saved
steps: 751.0
step 0: 3.0884735584259033
step 100: 1.5506771802902222
step 200: 1.4513400793075562


In [41]:
import json

dense_runs = json.load(open('dense_runs.json'))

X_data = np.array([])
y_data = np.array([])
for i in dense_runs.keys():
    for j in dense_runs[i]:
        
        tmp = np.array([j['tokens'], j['params']])

        X_data = np.concatenate((X_data, tmp))

        y_data = np.append(y_data, np.array([j['train_loss']]))
        
X_data = X_data.reshape(len(X_data)//2, 2)

In [42]:
from scipy.optimize import minimize
from scipy.special import huber
import numpy as np
import math


def fun(x, X_data, y_data):
    A, B, E, a, b = x
    losses=[]
    for i,j in zip(X_data, y_data):
        N,D = i
        target = j

        pred = E + (A/(N**a)) + (B/(D**b))

        r = math.log(pred)-math.log(target)
        

        loss = huber(10**(-3), r)
        losses.append(loss)

    return sum(losses)/len(losses)


res = minimize(fun, x0=np.array([1,1,1,1,1]), args=(X_data, y_data), method='L-BFGS-B')


In [45]:
A, B, E, a, b = res.x

G = ((a*A)/(b*B))**(1/(a+b))

exponent_n = b/(a+b)

exponent_d = a/(a+b)


print(f"G: {G}")
print(f"exponent_n: {exponent_n}")
print(f"exponent_d: {exponent_d}")
print(f"a: {a}")
print(f"b: {b}")
print(f"A: {A}")
print(f"B: {B}")
print(f"E: {E}")


G: 0.9998198264461936
exponent_n: 0.5001000939531397
exponent_d: 0.49989990604686024
a: 0.999579261513157
b: 0.9999795490049174
A: 1.0000415932957245
B: 1.0000015166901735
E: 1.3056251492960775


In [46]:
import json

dense_runs = json.load(open('moe_runs.json'))

X_data = np.array([])
y_data = np.array([])
for i in dense_runs.keys():
    for j in dense_runs[i]:
        
        tmp = np.array([j['tokens'], j['params']])

        X_data = np.concatenate((X_data, tmp))

        y_data = np.append(y_data, np.array([j['train_loss']]))
        
X_data = X_data.reshape(len(X_data)//2, 2)

In [47]:
from scipy.optimize import minimize
from scipy.special import huber
import numpy as np
import math


def fun(x, X_data, y_data):
    A, B, E, a, b = x
    losses=[]
    for i,j in zip(X_data, y_data):
        N,D = i
        target = j

        pred = E + (A/(N**a)) + (B/(D**b))

        r = math.log(pred)-math.log(target)
        

        loss = huber(10**(-3), r)
        losses.append(loss)

    return sum(losses)/len(losses)


res = minimize(fun, x0=np.array([1,1,1,1,1]), args=(X_data, y_data), method='L-BFGS-B')


In [48]:
A, B, E, a, b = res.x

G = ((a*A)/(b*B))**(1/(a+b))

exponent_n = b/(a+b)

exponent_d = a/(a+b)


print(f"G: {G}")
print(f"exponent_n: {exponent_n}")
print(f"exponent_d: {exponent_d}")
print(f"a: {a}")
print(f"b: {b}")
print(f"A: {A}")
print(f"B: {B}")
print(f"E: {E}")


G: 0.9999411606819287
exponent_n: 0.5000326630701785
exponent_d: 0.49996733692982165
a: 0.9998672525114933
b: 0.9999978959828785
A: 1.0000131038994298
B: 1.0000001256962225
E: 1.2321641154491534
